# 09 — Collider Signatures

**SDGFT predictions for high-energy particle colliders.**

The dimensional flow D(k) = D* + (4 − D*) · (k/M_Pl)^{−2γ_geo} modifies
scattering amplitudes at high √s:

| # | Channel | SDGFT effect | Testable at |
|---|---------|-------------|-------------|
| 1 | Running couplings | Modified β-functions | FCC-ee (Z-pole) |
| 2 | Drell-Yan | Enhanced tail at high m_ll | LHC Run 3 |
| 3 | Graviton exchange | Virtual spin-2 | FCC-hh |
| 4 | KK-like modes | Effective tower from D* > 2 | n/a (too heavy) |
| 5 | Higgs gg→H | Modified top loop | HL-LHC |
| 6 | Dijet angular | Contact interaction | LHC Run 3 |

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
import math

from sdgft_ml.physics import constants as C
from sdgft_ml.physics import dimension as D
from sdgft_ml.physics import collider, rg_running

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
print("Imports OK")
print(f"  D* = {D.D_STAR_TREE_F:.6f}")
print(f"  γ²_geo = {D.GAMMA_GEO_TREE_SQ_F:.2e}")
print(f"  M_Pl = {C.M_PL_GEV:.3e} GeV")

---
## 1  Running Couplings to Planck Scale

SM 1-loop running with SDGFT correction to beta functions:
$$b_i \to b_i \cdot (1 + c_i \cdot \gamma_{\text{geo}}^2)$$

In [ ]:
# Energy scan from M_Z to Planck scale
energies = np.logspace(np.log10(91.2), 19, 500)

# SM running
sm_results = [rg_running.run_to_scale(math.log(e / rg_running.M_Z)) for e in energies]
ia1_sm = [r["inv_alpha_1"] for r in sm_results]
ia2_sm = [r["inv_alpha_2"] for r in sm_results]
ia3_sm = [r["inv_alpha_3"] for r in sm_results]

# SDGFT-modified running
sdgft_results = [collider.sdgft_modified_running(e) for e in energies]
ia1_sd = [r["inv_alpha_1"] for r in sdgft_results]
ia2_sd = [r["inv_alpha_2"] for r in sdgft_results]
ia3_sd = [r["inv_alpha_3"] for r in sdgft_results]

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogx(energies, ia1_sm, "r--", lw=1, alpha=0.5)
ax.semilogx(energies, ia2_sm, "b--", lw=1, alpha=0.5)
ax.semilogx(energies, ia3_sm, "g--", lw=1, alpha=0.5)
ax.semilogx(energies, ia1_sd, "r-", lw=2, label="α₁⁻¹ (SDGFT)")
ax.semilogx(energies, ia2_sd, "b-", lw=2, label="α₂⁻¹ (SDGFT)")
ax.semilogx(energies, ia3_sd, "g-", lw=2, label="α₃⁻¹ (SDGFT)")

ax.axvline(C.M_PL_GEV, color="gray", ls=":", alpha=0.5, label="M_Pl")
ax.set_xlabel("√s  (GeV)")
ax.set_ylabel("1/α_i")
ax.set_title("Gauge Coupling Running: SM (dashed) vs SDGFT (solid)")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_ylim(0, 70)
plt.tight_layout()
plt.show()

# Difference at M_Pl
sm_pl = rg_running.run_to_scale(math.log(C.M_PL_GEV / rg_running.M_Z))
sd_pl = collider.sdgft_modified_running(C.M_PL_GEV)
print(f"At M_Pl:")
print(f"  SM:    α₁⁻¹={sm_pl['inv_alpha_1']:.2f}, α₂⁻¹={sm_pl['inv_alpha_2']:.2f}, α₃⁻¹={sm_pl['inv_alpha_3']:.2f}")
print(f"  SDGFT: α₁⁻¹={sd_pl['inv_alpha_1']:.2f}, α₂⁻¹={sd_pl['inv_alpha_2']:.2f}, α₃⁻¹={sd_pl['inv_alpha_3']:.2f}")

---
## 2  Modified Drell-Yan Cross-Section

At high invariant mass $m_{\ell\ell}$, the D*-dimensional propagator enhances the tail:

$$\frac{\sigma_{\text{SDGFT}}}{\sigma_{\text{SM}}} = 1 + (D^*-3)\,\frac{\alpha}{\pi}\,\ln\frac{m_{\ell\ell}}{M_Z}$$

In [ ]:
# Drell-Yan ratio vs invariant mass
m_ll = np.logspace(2, 4.5, 200)  # 100 GeV to ~30 TeV
dy_ratio = [collider.drell_yan_ratio(m) for m in m_ll]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")

# Left: ratio
ax1.semilogx(m_ll / 1000, dy_ratio, "b-", lw=2)
ax1.axhline(1.0, color="gray", ls="--", alpha=0.5)
ax1.set_xlabel("m_ℓℓ  (TeV)")
ax1.set_ylabel("σ_SDGFT / σ_SM")
ax1.set_title("Drell-Yan Cross-Section Ratio")
ax1.grid(alpha=0.3)

# Right: deviation in %
deviation_pct = [(r - 1) * 100 for r in dy_ratio]
ax2.semilogx(m_ll / 1000, deviation_pct, "r-", lw=2)
ax2.set_xlabel("m_ℓℓ  (TeV)")
ax2.set_ylabel("Deviation (%)")
ax2.set_title("SDGFT Deviation from SM Drell-Yan")
ax2.grid(alpha=0.3)

# LHC / FCC-hh reach lines
for ax in [ax1, ax2]:
    ax.axvline(3.0, color="orange", ls=":", alpha=0.5)
    ax.text(3.0, 0.95, " LHC", fontsize=8, color="orange",
            transform=ax.get_xaxis_transform(), va="top")
    ax.axvline(15.0, color="purple", ls=":", alpha=0.5)
    ax.text(15.0, 0.95, " FCC-hh", fontsize=8, color="purple",
            transform=ax.get_xaxis_transform(), va="top")

plt.show()

print(f"DY deviation at m_ll = 3 TeV:  {(collider.drell_yan_ratio(3000) - 1)*100:.4f} %")
print(f"DY deviation at m_ll = 10 TeV: {(collider.drell_yan_ratio(10000) - 1)*100:.4f} %")

---
## 3  Virtual Graviton Exchange

In D* dimensions, the graviton exchange amplitude scales as:
$$\mathcal{A}_{\text{grav}} \sim \left(\frac{\sqrt{s}}{M_{\text{Pl}}}\right)^{D^*-2}$$

For D* ≈ 2.79: growth is **slower** than D=4 gravity ($\propto s/M_{\text{Pl}}^2$).

In [ ]:
# Graviton amplitude vs √s
sqrt_s_range = np.logspace(2, 19, 300)  # 100 GeV to M_Pl

amp_sdgft = [collider.graviton_exchange_amplitude(e) for e in sqrt_s_range]
amp_d4 = [(e / C.M_PL_GEV)**2 for e in sqrt_s_range]  # Standard D=4

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(sqrt_s_range, amp_sdgft, "r-", lw=2, label=f"SDGFT (D*={D.D_STAR_TREE_F:.3f})")
ax.loglog(sqrt_s_range, amp_d4, "b--", lw=1.5, label="D=4 gravity")
ax.axvline(14000, color="orange", ls=":", alpha=0.7, label="LHC 14 TeV")
ax.axvline(100000, color="purple", ls=":", alpha=0.7, label="FCC-hh 100 TeV")
ax.set_xlabel("√s  (GeV)")
ax.set_ylabel("|A_grav|")
ax.set_title("Virtual Graviton Exchange Amplitude")
ax.legend(fontsize=10)
ax.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print(f"A_grav at 14 TeV (LHC):    {collider.graviton_exchange_amplitude(14000):.2e}")
print(f"A_grav at 100 TeV (FCC):   {collider.graviton_exchange_amplitude(100000):.2e}")
print(f"σ_grav at 14 TeV:          {collider.graviton_exchange_cross_section_fb(14000):.2e} fb")
print(f"σ_grav at 100 TeV:         {collider.graviton_exchange_cross_section_fb(100000):.2e} fb")

---
## 4  Effective Kaluza-Klein Spectrum

Although D* < 3 means no true extra dimensions,
the fractal geometry produces a KK-like tower with suppressed couplings.

In [ ]:
# KK spectrum: default compactification at M_Pl · (D*-2)
modes = collider.kk_spectrum(n_max=10)

print("Effective KK spectrum:")
print(f"{'n':>3s} {'M_n (GeV)':>15s} {'coupling':>12s}")
print("─" * 32)
for m in modes:
    print(f"{m.n:3d} {m.mass_gev:15.3e} {m.coupling_ratio:12.3e}")

# Bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ns = [m.n for m in modes]
masses = [m.mass_gev for m in modes]
couplings = [m.coupling_ratio for m in modes]

ax1.bar(ns, masses, color="steelblue")
ax1.set_yscale("log")
ax1.set_xlabel("Mode number n")
ax1.set_ylabel("M_n (GeV)")
ax1.set_title("KK-like mass spectrum")

ax2.bar(ns, couplings, color="coral")
ax2.set_yscale("log")
ax2.set_xlabel("Mode number n")
ax2.set_ylabel("Coupling / zero-mode")
ax2.set_title("Relative coupling strength")

plt.tight_layout()
plt.show()

---
## 5  Higgs Production Modification

The gg→H loop picks up a geometric correction:
$$\frac{\sigma_{gg \to H}}{\sigma_{gg \to H}^{\text{SM}}} = 1 + 2\gamma_{\text{geo}}^2 \ln(m_t/m_b)$$

In [ ]:
# Higgs modifications
higgs_prod = collider.higgs_gg_modification()
higgs_width = collider.higgs_width_modification()

print("Higgs modifications (SDGFT):")
print(f"  σ(gg→H) / σ_SM  = {higgs_prod:.10f}")
print(f"  Deviation:         {(higgs_prod - 1)*1e6:.2f} ppm")
print(f"  Γ(H) / Γ_SM     = {higgs_width:.10f}")
print(f"  Deviation:         {(higgs_width - 1)*1e6:.2f} ppm")
print(f"\n  LHC precision:   ~5-10%")
print(f"  HL-LHC goal:     ~2-5%")
print(f"  FCC-ee goal:     ~0.5%")
print(f"  SDGFT deviation: ~{(higgs_prod - 1)*100:.1e}% — far below current reach")

# Scan D* for larger effects
d_scan = np.linspace(2.5, 3.5, 100)
higgs_scan = [collider.higgs_gg_modification(d) for d in d_scan]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(d_scan, [(h - 1) * 100 for h in higgs_scan], "b-", lw=2)
ax.axvline(D.D_STAR_TREE_F, color="red", ls=":", label=f"D* = {D.D_STAR_TREE_F:.3f}")
ax.axhline(0.5, color="green", ls="--", alpha=0.5, label="FCC-ee precision (0.5%)")
ax.axhline(5.0, color="orange", ls="--", alpha=0.5, label="HL-LHC precision (5%)")
ax.set_xlabel("D*")
ax.set_ylabel("Higgs gg→H deviation (%)")
ax.set_title("Higgs Production Modification vs D*")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6  Dijet Angular Distributions

SDGFT adds a spin-2 contact term to dijet scattering,
modifying the angular distribution $F(\chi)$.

In [ ]:
# F(χ) — dijet angular ratio
chi_range = np.linspace(1.1, 30, 200)
f_chi = [collider.dijet_f_chi(chi) for chi in chi_range]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(chi_range, f_chi, "r-", lw=2, label=f"SDGFT (D*={D.D_STAR_TREE_F:.3f})")
ax.axhline(1.0, color="gray", ls="--", alpha=0.5, label="SM (QCD only)")
ax.set_xlabel("χ = exp|y₁ − y₂|")
ax.set_ylabel("F(χ)_SDGFT / F(χ)_SM")
ax.set_title("Dijet Angular Distribution Ratio")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"F(χ=1.5)/F_SM = {collider.dijet_f_chi(1.5):.6f}  (enhancement at small angles)")
print(f"F(χ=30)/F_SM  = {collider.dijet_f_chi(30.0):.6f}  (effect at large angles)")

---
## 7  BSM Discovery Reach

Estimated reach for different collider configurations.

In [ ]:
# Reach estimates
configs = [
    ("LHC Run 3", 14.0, 300),
    ("HL-LHC", 14.0, 3000),
    ("HE-LHC", 27.0, 3000),
    ("FCC-hh", 100.0, 30000),
]

print("SDGFT BSM Reach Estimates:")
print("═" * 75)
for name, sqrt_s, lumi in configs:
    print(f"\n{name} (√s = {sqrt_s} TeV, L = {lumi} fb⁻¹):")
    reaches = collider.compute_reach(sqrt_s, lumi)
    for r in reaches:
        if "Higgs" in r.process:
            print(f"  {r.process:45s}  shift = {r.significance_sigma:.1f} ppm")
        elif "Drell-Yan" in r.process:
            print(f"  {r.process:45s}  deviation = {r.significance_sigma:.4f}%")
        else:
            print(f"  {r.process:45s}  M_reach = {r.m_reach_tev:.1f} TeV")

In [ ]:
# Energy scan — all observables at once
scan = collider.energy_scan()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), layout="constrained")

e_arr = [s["sqrt_s_gev"] for s in scan]

# 1. α_s
axes[0].semilogx(e_arr, [s["alpha_s"] for s in scan], "g-", lw=2)
axes[0].set_xlabel("√s (GeV)"); axes[0].set_ylabel("α_s")
axes[0].set_title("Strong coupling (SDGFT)"); axes[0].grid(alpha=0.3)

# 2. DY ratio
axes[1].semilogx(e_arr, [s["drell_yan_ratio"] for s in scan], "b-", lw=2)
axes[1].axhline(1.0, color="gray", ls="--", alpha=0.5)
axes[1].set_xlabel("√s (GeV)"); axes[1].set_ylabel("σ/σ_SM")
axes[1].set_title("Drell-Yan ratio"); axes[1].grid(alpha=0.3)

# 3. Graviton amplitude
axes[2].loglog(e_arr, [s["graviton_amplitude"] for s in scan], "r-", lw=2)
axes[2].set_xlabel("√s (GeV)"); axes[2].set_ylabel("|A_grav|")
axes[2].set_title("Graviton amplitude"); axes[2].grid(alpha=0.3)

fig.suptitle("SDGFT Collider Observable Energy Scan", fontsize=13)
plt.show()

---
## Summary

| Observable | SDGFT deviation | Current precision | Testable? |
|-----------|----------------|------------------|----------|
| Drell-Yan (3 TeV) | ~0.04% | ~3% (LHC) | Not yet |
| Higgs gg→H | ~10⁻⁴ ppm | ~5% | No (need 10⁹× improvement) |
| Graviton exchange | 10⁻²² fb | — | No (suppressed by M_Pl) |
| Dijet angular | ~10⁻⁴⁴ | ~1% | No |
| Running couplings | ~γ²_geo | per-mille (Z-pole) | FCC-ee prospect |

**Key insight**: SDGFT effects at collider energies are extremely small because
$\gamma_{\text{geo}}^2 \approx 4 \times 10^{-7}$.  The theory is naturally consistent
with all collider null results. Falsification route: precision FCC-ee measurements.